# Employee Attrition & Financial Risk Analytics
## Notebook 1 of 3 — Data Cleaning & Exploratory Data Analysis

**Project:** Employee Risk Analytics
**Dataset:** IBM HR Analytics Employee Attrition dataset (1,470 employees, 35 columns)

**Goal of this notebook:** load the raw HR dataset, clean it, engineer a few analysis-friendly
groupings (salary bands, age bands), and explore which factors are most associated with employee
attrition.

**Pipeline position:** this is the first of three notebooks in the project.

| Notebook | Purpose |
|---|---|
| **1 — Data Cleaning & EDA (this notebook)** | Load, clean, explore |
| 2 — Modeling & Financial Risk | Encode, train models, score financial risk |
| 3 — Explainability & Dashboards | SHAP explainability, Power BI exports |

**Output of this notebook:** `data/employee_attrition_clean.csv`, the single cleaned and
feature-engineered dataset that Notebook 2 loads to start modeling.


## 1. Setup

Standard imports and a local project folder structure (`data/`, `results/`, `visualizations/`)
so the notebook runs the same way on any machine — no Google Drive mount required.


In [ ]:
import os
import pandas as pd

# Local project folders (created if missing) — keeps every notebook in this
# project runnable from a fresh clone with no cloud-drive dependency.
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("visualizations", exist_ok=True)

pd.set_option('display.max_columns', None)


## 2. Load the Raw Dataset

> **Input required:** place `WA_Fn-UseC_-HR-Employee-Attrition.csv` (IBM HR Analytics Employee
> Attrition dataset) inside a `data/` folder next to this notebook before running.


In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
df.head()


In [ ]:
df.shape


In [ ]:
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")


In [ ]:
df.info()


In [ ]:
df['Attrition'].value_counts()


**Finding:** 1,233 employees stayed and 237 left — an overall attrition rate of about 16.1%.
This is a moderately imbalanced target, worth keeping in mind when evaluating classifiers later.


In [ ]:
pd.set_option('display.max_columns', None)
df.head()


## 3. Data Quality Checks

Checking for missing values, duplicate rows, and the cardinality/type of each column before
deciding what to clean or engineer.


In [ ]:
df.isnull().sum()


In [ ]:
df.duplicated().sum()


**Finding:** no missing values and no duplicate rows — the dataset is already clean at the row
level. The cleaning step below is about removing *uninformative* columns, not fixing bad data.


In [ ]:
df.nunique()


In [ ]:
df.select_dtypes(include=['int64']).columns


In [ ]:
df.select_dtypes(include=['object']).columns


## 4. Data Cleaning

Four columns carry no analytical value:
- `EmployeeCount` and `StandardHours` — constant across all 1,470 rows
- `EmployeeNumber` — just a row identifier
- `Over18` — constant (every record is 'Y')

These are dropped from a working copy, `df_clean`, so the original `df` stays untouched as a
reference.


In [ ]:
df_clean = df.copy()
df_clean.shape


In [ ]:
columns_to_drop = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df_clean = df_clean.drop(columns=columns_to_drop)


In [ ]:
df_clean.shape


In [ ]:
df_clean.columns


In [ ]:
df_clean.info()


In [ ]:
df_clean.head()


In [ ]:
df_clean.describe()


## 5. Exploratory Data Analysis

### 5.1 Overall Attrition Distribution


In [ ]:
df_clean['Attrition'].value_counts()


In [ ]:
round(df_clean['Attrition'].value_counts(normalize=True) * 100,2)


In [ ]:
import matplotlib.pyplot as plt

ax = df_clean['Attrition'].value_counts().plot(
    kind='bar',
    color=['green','red'],
    figsize=(6,5))

plt.title('Employee Attrition Distribution')
plt.xlabel('Attrition')
plt.ylabel('Number of Employees')

for container in ax.containers:
  ax.bar_label(container)

plt.show()



### 5.2 Attrition by Department


In [ ]:
pd.crosstab(
    df_clean['Attrition'],
    df_clean['Department']
)


In [ ]:
dept_attrition = pd.crosstab(
    df_clean['Department'],
    df_clean['Attrition'],
    normalize='index'
) *100

dept_attrition


In [ ]:
ay = dept_attrition['Yes'].sort_values(
    ascending=True
).plot(
    kind = 'barh',
    color = ['yellow','orange','red'],
    figsize = (12,4)
)

for container in ay.containers:
  ay.bar_label(container, fmt= '%.2f%%')

plt.title('Department')
plt.xlabel('Attrition Rate by Department')
plt.ylabel('Attrition Rate (%)')

plt.xticks(rotation=0)

plt.show()


**Finding:** Sales has the highest attrition rate, followed by R&D, with HR the lowest.


### 5.3 Attrition by Salary Group

`MonthlyIncome` is used to derive an `EstimatedMonthlySalary` figure, which is then bucketed into
quartile-based `SalaryGroup` bands (Low / Medium / High / Very High) for segment-level analysis.


In [ ]:
df_clean['MonthlyIncome'].describe()


In [ ]:
df_clean['EstimatedMonthlySalary'] = df_clean['MonthlyIncome'] * 10
df_clean.head()



In [ ]:
df_clean['EstimatedMonthlySalary'].describe()


In [ ]:
df_clean['SalaryGroup'] = pd.qcut(
    df_clean['EstimatedMonthlySalary'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)


In [ ]:
df_clean['SalaryGroup'].value_counts()


In [ ]:
salary_attrition = pd.crosstab(
    df_clean['SalaryGroup'],
    df_clean['Attrition'],
    normalize='index'
) * 100

salary_attrition


In [ ]:
ax = salary_attrition['Yes'].sort_values().plot(
    kind = 'barh',
    color = ['blue','blue','blue','red'],
    figsize = (8,4)
)

for container in ax.containers:
  ax.bar_label(container, fmt = '%.1f%%')

plt.title('Employe Attrition by Salary Group')
plt.xlabel('Attrition rate %')
plt.ylabel('Salary Group')

plt.grid(axis = 'x', linestyle = '--', alpha = 0.5)
plt.tight_layout()
plt.show()


**Finding:** attrition drops sharply as salary rises — the Low salary group leaves at roughly
3x the rate of the Very High group.


### 5.4 Attrition by Age Group

`Age` is bucketed into four bands for the same kind of segment analysis.


In [ ]:
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins = [18,25,35,45,60],
    labels = [
        '18-25',
        '26-35',
        '36-45',
        '46-60'

    ],
    include_lowest = True
)

df_clean['AgeGroup'].value_counts()


In [ ]:
age_attrition = pd.crosstab(
    df_clean['AgeGroup'],
    df_clean['Attrition'],
    normalize='index'
    ) * 100

age_attrition


In [ ]:
ax1 = age_attrition['Yes'].plot(
    kind = 'bar',
    color = ['red','red','blue','blue'],
    figsize= [8,4]
)

for container in ax1.containers:
  ax1.bar_label(container, fmt = '%.2f%%')

plt.tight_layout()

plt.title('Attrition by Age')
plt.xlabel('Age Group')
plt.ylabel('Attrition Rate %')
plt.show()


**Finding:** the youngest band (18–25) attrites at nearly 4x the rate of the 36–45 band —
younger employees are clearly the highest-risk age segment.


### 5.5 Attrition by Overtime


In [ ]:
overtime_attrition = pd.crosstab(
    df_clean['OverTime'],
    df_clean['Attrition'],
    normalize='index'
) *100

overtime_attrition


In [ ]:
ax = overtime_attrition['Yes'].plot(
    color = ['#F4A6A6', '#8B0000'],
    kind='bar',
    figsize=(6,4)
)

for container in ax.containers:
  ax.bar_label(container, fmt= '%.1f%%')

plt.xlabel('Overtime')
plt.ylabel('Attrition rate')
plt.title('Attrition by Overtime')

plt.show()


**Finding:** employees who work overtime leave at roughly 3x the rate of those who don't —
one of the strongest single predictors of attrition in this dataset.


### 5.6 Attrition by Job Satisfaction


In [ ]:
jobsat_attrition = pd.crosstab(
    df['JobSatisfaction'],
    df['Attrition'],
    normalize = 'index'
) *100

jobsat_attrition


In [ ]:
ax = jobsat_attrition['Yes'].plot(
    kind = 'barh',
    color = [
    '#8B0000',  # Dark Red
    '#CD5C5C',  # Indian Red
    '#F08080',  # Light Coral
    '#FFE5E5'   # Very Light Red
],
    figsize = [8,4]
)

for container in ax.containers:
  ax.bar_label(container, fmt = '%.2f%%')


  plt.xlabel('Attrition Rate')
  plt.ylabel('Job Satisfaction')
  plt.title('Attrition by Job Satisfaction')


**Finding:** attrition decreases steadily as job satisfaction rises (level 1 → level 4),
though the relationship is weaker than the salary or overtime effects above.


## 6. Correlation Analysis

To see how all numeric features relate to attrition at once, `Attrition` is mapped to a numeric
`AttritionNumeric` flag (1 = left, 0 = stayed) and correlated against every other numeric column.


In [ ]:
df_clean['AttritionNumeric'] = df_clean['Attrition'].map(
    {
        'Yes': 1,
        'No' : 0
    }
)

df_clean[['AttritionNumeric','Attrition']].head()


In [ ]:
correlation_with_attrition = df_clean.corr(
    numeric_only = True
)['AttritionNumeric'].sort_values(ascending = False)

correlation_with_attrition


In [ ]:
import seaborn as sns

plt.figure(figsize = (14,10))

sns.heatmap(
    df_clean.corr(
        numeric_only = True
    ),
    annot = True,
    fmt = '.2f',
    cmap = 'coolwarm',
    center=0,
    linewidths = 0.5
)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


**Finding:** no single numeric feature correlates strongly with attrition in isolation
(`TotalWorkingYears`, `JobLevel`, and `MonthlyIncome` show the strongest negative correlations).
This is expected — attrition here is driven by a *combination* of factors, which is exactly why
a machine learning model (Notebook 2) is more useful than any single crosstab.


## 7. Save the Cleaned, Feature-Engineered Dataset

This is the single output of this notebook: `df_clean` now includes the engineered
`EstimatedMonthlySalary`, `SalaryGroup`, `AgeGroup`, and `AttritionNumeric` columns on top of the
original cleaned fields. **Notebook 2 loads this file to begin modeling** — this is the dependency
handoff between notebooks 1 and 2.


In [ ]:
df_clean.info()


In [ ]:
df_clean.to_csv('data/employee_attrition_clean.csv', index=False)
print("Saved cleaned dataset -> data/employee_attrition_clean.csv")
print("Shape:", df_clean.shape)


---
**Next:** open `02_modeling_and_financial_risk.ipynb` to encode features, train and compare
Logistic Regression and Random Forest models, and translate predictions into financial risk.
